# Получение доходов консолидированного бюджета РФ через API «Электронного бюджета»

В этом блокноте проверяется подключение к API Единого портала бюджетной системы РФ и извлекаются годовые поступления доходов консолидированного бюджета за 2020–2022 годы. Полученные значения используются для расчёта макроэкономической нагрузки налоговой задолженности.


## Шаг 1. Проверка подключения и структуры ответа API
Загружаем адрес API из файла `.env`, отправляем тестовый запрос за 2022 год и проверяем доступные поля JSON-ответа.


In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
raw_url = os.getenv("BUDGET_API_URL")
base_url = raw_url.split("?")[0] if raw_url else ""

params = {
    "filterperiod": "01012023",
    "filterperiodicity": "Y",
    "pageSize": 5
}

response = requests.get(base_url, params=params, timeout=15)
print("Статус ответа:", response.status_code)

payload = response.json()
print("Всего записей за период:", payload.get("recordCount"))

records = payload.get("data", [])
df_test = pd.DataFrame(records)
print("\nНазвания полей в ответе API:")
print(df_test.columns.tolist())
print("\nПервые 3 строки:")
print(df_test.head(3))


Статус ответа: 200
Всего записей за период: 2738

Названия полей в ответе API:
['period', 'periodicity', 'blockSubbo', 'blockDatasource', 'blockKd', 'blockStrcode', 'strcode', 'kvsr', 'kd', 'perkb', 'pertf', 'exkbrfgvf', 'exfb', 'exgvf', 'guid', 'typereport']

Первые 3 строки:
     period periodicity                                         blockSubbo  \
0  01012023           Y  {'foname': 'Федеральное казначейство', 'focode...   
1  01012023           Y  {'foname': 'Федеральное казначейство', 'focode...   
2  01012023           Y  {'foname': 'Федеральное казначейство', 'focode...   

                                   blockDatasource  \
0  {'id': '7', 'name': 'Федеральное казначейство'}   
1  {'id': '7', 'name': 'Федеральное казначейство'}   
2  {'id': '7', 'name': 'Федеральное казначейство'}   

                                             blockKd  \
0  {'id': '15837', 'code': '20225359000000150', '...   
1  {'id': '15833', 'code': '20225353000000150', '...   
2  {'id': '15325', 'code

## Шаг 2. Поиск укрупнённых кодов бюджетной классификации (КБК)
Фильтруем записи по строке доходов (`strcode == 010`) и находим итоговые групповые КБК, чтобы определить код совокупных налоговых и неналоговых доходов.


In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
raw_url = os.getenv("BUDGET_API_URL")
base_url = raw_url.split("?")[0] if raw_url else ""

# Забираем все 2738 записей за 2022 год одним запросом
params = {
    "filterperiod": "01012023",
    "filterperiodicity": "Y",
    "pageSize": 3000
}

response = requests.get(base_url, params=params, timeout=30)
records = response.json().get("data", [])
df = pd.DataFrame(records)

# Оставляем только строки доходов (strcode == '010')
df_inc = df[df["strcode"] == "010"].copy()
df_inc["kd_name"] = df_inc["blockKd"].apply(lambda x: x.get("name") if isinstance(x, dict) else "")

# Ищем укрупненные налоговые КБК (где код заканчивается на '00000000000')
agg_tax = df_inc[df_inc["kd"].str.endswith("00000000000")].copy()

print(f"Найдено укрупненных групп доходов: {len(agg_tax)}\n")
for _, row in agg_tax[["kd", "kd_name", "perkb", "exkbrfgvf"]].iterrows():
    # Переводим рубли в тыс. рублей (делим на 1000), чтобы совпадало с формой 4-НМ
    kb_tys = float(row["perkb"]) / 1000
    gvf_tys = float(row["exkbrfgvf"]) / 1000
    print(f"КБК: {row['kd']} | Конс.бюджет (тыс.руб): {kb_tys:,.0f} | С фондами: {gvf_tys:,.0f} | {row['kd_name'][:55]}")


Найдено укрупненных групп доходов: 14

КБК: 11413000000000000 | Конс.бюджет (тыс.руб): 10,845,998 | С фондами: 11,780,382 | Доходы от приватизации имущества, находящегося в госуда
КБК: 20300000000000000 | Конс.бюджет (тыс.руб): 210,497,445 | С фондами: 212,014,239 | БЕЗВОЗМЕЗДНЫЕ ПОСТУПЛЕНИЯ ОТ ГОСУДАРСТВЕННЫХ (МУНИЦИПАЛ
КБК: 11200000000000000 | Конс.бюджет (тыс.руб): 49,853,490 | С фондами: 531,136,961 | ПЛАТЕЖИ ПРИ ПОЛЬЗОВАНИИ ПРИРОДНЫМИ РЕСУРСАМИ
КБК: 10000000000000000 | Конс.бюджет (тыс.руб): 15,432,733,607 | С фондами: 52,690,649,649 | НАЛОГОВЫЕ И НЕНАЛОГОВЫЕ ДОХОДЫ
КБК: 20000000000000000 | Конс.бюджет (тыс.руб): 4,244,176,161 | С фондами: 383,539,139 | БЕЗВОЗМЕЗДНЫЕ ПОСТУПЛЕНИЯ
КБК: 20700000000000000 | Конс.бюджет (тыс.руб): 38,273,389 | С фондами: 39,039,039 | ПРОЧИЕ БЕЗВОЗМЕЗДНЫЕ ПОСТУПЛЕНИЯ
КБК: 11800000000000000 | Конс.бюджет (тыс.руб): 6,438 | С фондами: 6,416 | ПОСТУПЛЕНИЯ (ПЕРЕЧИСЛЕНИЯ) ПО УРЕГУЛИРОВАНИЮ РАСЧЕТОВ М
КБК: 10600000000000000 | Конс.бюджет (тыс.руб): 1,632,241,

## Шаг 3. Выгрузка годовых доходов консолидированного бюджета за 2020–2022 гг.
Извлекаем итоговые поступления по КБК `10000000000000000` за каждый год и переводим суммы из рублей в тысячи рублей для сопоставимости с данными ФНС.


In [3]:
# Ищем нужные налоги по тексту названия и оставляем только укрупненные строки (заканчиваются на '00000')
keywords = "добавленную стоимость|прибыль организаций|добычу полезных ископаемых"
found_tax = df_inc[
    df_inc["kd_name"].str.contains(keywords, case=False, na=False) & 
    df_inc["kd"].str.endswith("00000")
].copy()

print(f"Найдено строк: {len(found_tax)}\n")
for _, row in found_tax[["kd", "kd_name", "exkbrfgvf", "exfb"]].head(15).iterrows():
    total_tys = float(row["exkbrfgvf"]) / 1000
    fb_tys = float(row["exfb"]) / 1000
    print(f"КБК: {row['kd']} | Всего (тыс.руб): {total_tys:,.0f} | Фед.бюджет: {fb_tys:,.0f} | {row['kd_name'][:55]}")


Найдено строк: 0



## Шаг 4. Диагностика отсутствующего итогового КБК за 2020 год
Проверяем состав записей за 2020 год (период `01012021`) и убеждаемся, что в открытом наборе API за этот год отсутствует агрегированная строка по всему консолидированному бюджету.


In [4]:
# Ищем по коротким фрагментам слов без ограничения на нули в КБК
short_keys = "НДС|добавленн|прибыл|ископаем"
found_any = df_inc[df_inc["kd_name"].str.contains(short_keys, case=False, na=False)].copy()

print(f"Найдено строк: {len(found_any)}\n")
for _, row in found_any[["kd", "kd_name", "exkbrfgvf", "exfb"]].head(15).iterrows():
    total_tys = float(row["exkbrfgvf"]) / 1000
    fb_tys = float(row["exfb"]) / 1000
    print(f"КБК: {row['kd']} | Всего (тыс.руб): {total_tys:,.0f} | Фед.бюджет: {fb_tys:,.0f} | {row['kd_name'][:55]}")


Найдено строк: 31

КБК: 10101024010000110 | Всего (тыс.руб): 4,485,576 | Фед.бюджет: 1,121,394 | Налог на прибыль организаций при выполнении соглашений 
КБК: 10702030010000110 | Всего (тыс.руб): 1 | Фед.бюджет: 1 | Регулярные платежи за добычу полезных ископаемых (роялт
КБК: 10401000010000110 | Всего (тыс.руб): 3,063,519,040 | Фед.бюджет: 3,063,419,640 | Налог на добавленную стоимость на товары, ввозимые на т
КБК: 10901020110000110 | Всего (тыс.руб): -8 | Фед.бюджет: 0 | Налог на прибыль организаций, зачислявшийся до 1 января
КБК: 10101070010000110 | Всего (тыс.руб): 151,769,841 | Фед.бюджет: 151,769,841 | Налог на прибыль организаций с доходов, полученных в ви
КБК: 10702010010000110 | Всего (тыс.руб): 7,896,481 | Фед.бюджет: 7,896,481 | Регулярные платежи за добычу полезных ископаемых (роялт
КБК: 10101030010000110 | Всего (тыс.руб): 33,234,956 | Фед.бюджет: 33,234,956 | Налог на прибыль организаций с доходов иностранных орга
КБК: 11405030010000440 | Всего (тыс.руб): 21,445,936 | Фед.б

In [5]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
raw_url = os.getenv("BUDGET_API_URL")
base_url = raw_url.split("?")[0] if raw_url else ""

periods = {
    2020: "01012021",
    2021: "01012022",
    2022: "01012023"
}

budget_rows = []

for year, period_str in periods.items():
    params = {
        "filterperiod": period_str,
        "filterperiodicity": "Y",
        "pageSize": 3000
    }
    response = requests.get(base_url, params=params, timeout=30)
    records = response.json().get("data", [])
    df = pd.DataFrame(records)
    
    # Фильтруем строку доходов (strcode == '010') и итоговый КБК налоговых и неналоговых доходов
    target = df[(df["strcode"] == "010") & (df["kd"] == "10000000000000000")]
    
    if not target.empty:
        row = target.iloc[0]
        # Переводим из рублей в тыс. рублей (делим на 1000) для совпадения с формой 4-НМ
        perkb_tys = round(float(row["perkb"]) / 1000, 1)
        budget_rows.append({
            "Год": year,
            "Отчетная_дата": period_str,
            "КБК": row["kd"],
            "Поступило_доходов_КБ_РФ_тыс_руб": perkb_tys
        })
    else:
        print(f"Внимание: за {year} год строка не найдена")

df_budget = pd.DataFrame(budget_rows)
print(df_budget.to_string(index=False))


Внимание: за 2020 год строка не найдена
 Год Отчетная_дата               КБК  Поступило_доходов_КБ_РФ_тыс_руб
2021      01012022 10000000000000000                     1.365177e+10
2022      01012023 10000000000000000                     1.543273e+10


In [6]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
raw_url = os.getenv("BUDGET_API_URL")
base_url = raw_url.split("?")[0] if raw_url else ""

params = {
    "filterperiod": "01012021",
    "filterperiodicity": "Y",
    "pageSize": 3000
}

response = requests.get(base_url, params=params, timeout=30)
payload = response.json()
print("Всего записей за 2020 год (01012021):", payload.get("recordCount"))

records = payload.get("data", [])
if records:
    df_2020 = pd.DataFrame(records)
    # Ищем любые строки, где в КБК встречается '10000000'
    found = df_2020[df_2020["kd"].astype(str).str.contains("10000000", na=False)]
    print(found[["strcode", "kd", "perkb", "exkbrfgvf"]].head(10).to_string(index=False))


Всего записей за 2020 год (01012021): 2455
strcode                kd            perkb        exkbrfgvf
    010 11202100000000120      21532338.98      77565212.48
    010 11401000000000410    8705294528.02    8706628528.07
    010 10901000000000110       2894207.01       2894207.01
    010 11406010000000430   20334115995.13   20334115995.13
    010 10101010000000110 2851738786307.56 3367753124889.02
    010 11105010000000120  159335887082.88  159335887082.88
    010 11301000000000130   15483276310.57  102633371806.51
    010 11000000000000000                0 1935184631326.04
    010 11101000000000120   20849456240.78  443517055799.78
    010 11105410000000120        1344723.6        1344723.6


In [7]:
# Ищем в уже загруженной таблице df_2020 строки доходов (strcode == 010) с самыми крупными суммами
df_2020_inc = df_2020[df_2020["strcode"] == "010"].copy()
df_2020_inc["kd_name"] = df_2020_inc["blockKd"].apply(lambda x: x.get("name") if isinstance(x, dict) else "")
df_2020_inc["perkb_num"] = pd.to_numeric(df_2020_inc["perkb"], errors="coerce")

# Сортируем по убыванию суммы в консолидированный бюджет и выводим топ-5 строк
top_2020 = df_2020_inc.sort_values(by="perkb_num", ascending=False).head(5)

for _, row in top_2020.iterrows():
    kb_tys = row["perkb_num"] / 1000
    print(f"КБК: '{row['kd']}' | Конс.бюджет (тыс.руб): {kb_tys:,.1f} | Название: {row['kd_name']}")


КБК: '10101010000000110' | Конс.бюджет (тыс.руб): 2,851,738,786.3 | Название: Налог на прибыль организаций, зачисляемый в бюджеты бюджетной системы Российской Федерации по соответствующим ставкам
КБК: '10101012020000110' | Конс.бюджет (тыс.руб): 2,361,884,360.6 | Название: Налог на прибыль организаций (за исключением консолидированных групп налогоплательщиков), зачисляемый в бюджеты субъектов Российской Федерации
КБК: '10600000000000000' | Конс.бюджет (тыс.руб): 1,358,026,455.6 | Название: НАЛОГИ НА ИМУЩЕСТВО
КБК: '10602000020000110' | Конс.бюджет (тыс.руб): 910,072,228.4 | Название: Налог на имущество организаций
КБК: '20240000000000150' | Конс.бюджет (тыс.руб): 853,802,299.3 | Название: Иные межбюджетные трансферты
